# 多模态
## 1. 什么是多模态
多模态是指模型能够同时理解和处理 不止一种类型的数据（模态），例如：
- 文本（Text）
- 图像（Vision）
- 音频（Audio）
- 视频（Video）
- 传感器数据（Sensor / IoT 数据）

人类在感知和交流时是天然的多模态，比如我们读书时同时看到文字和插图，听别人讲话还能看到对方的表情。

## 2. 传统模型
传统的 AI 模型往往只处理单一模态：
- NLP 模型 只会处理文本。
- CV 模型 只会处理图像。
- ASR/TTS 模型 只会处理音频。

而多模态模型可以融合多种输入信息，实现更强的理解和生成能力。
例如：
- 图文混合理解：问模型“这张图片中有几只猫？”
- 文生图：输入文字提示，生成符合语义的图片（Stable Diffusion）
- 看图回答问题：上传一张图，提问并得到基于图片内容的答案
- 视频理解：给视频自动生成摘要或标签
- 音视频分析：例如自动识别视频中的语音并关联画面内容

## 3. 任务分类
1. Audio-Text-to-Text (音频-文本到文本)：将音频（如语音）和辅助文本结合作为输入，生成文本输出。例如：语音转录时结合上下文文本提升准确性。
2. Image-Text-to-Text (图像-文本到文本)：结合图像和文本输入生成文本输出。例如：图像描述生成（输入图片+提示文本，输出详细描述）。
3. Visual Question Answering (视觉问答，VQA)：对给定的图像和自然语言问题生成答案。例如：输入图片+“这是什么动物？”，输出“狗”。
4. Document Question Answering (文档问答)：基于文档（如PDF、扫描件）中的文本或布局信息回答问题。例如：从发票中提取金额或日期。
5. Video-Text-to-Text (视频-文本到文本)：结合视频帧序列和文本输入生成文本输出。例如：视频内容摘要或基于视频的对话生成。
6. Visual Document Retrieval (视觉文档检索)：通过图像或文本查询搜索相关文档。例如：上传表格图片，检索数据库中的匹配文档。
7. Any-to-Any (任意模态到任意模态)：支持任意输入和输出模态的通用模型。例如：输入音频+图片，输出文本；或输入文本，生成图片+语音。

## 4. 常用模型
| 任务 | 输入 | 	输出 |	代表模型 |
| --- | --- | --- | --- |
| 图像描述 (Image Captioning)|	图像|	文字描述|	BLIP、OFA|
|视觉问答 (Visual Question Answering, VQA)|	图像 + 文字问题|	文字答案|	LLaVA、MiniGPT-4|
|文生图 (Text-to-Image)|	文字|	图像	|DALL·E 3、Stable Diffusion|
|图像搜索 (Text-Image Retrieval)|	文字或图像	|相似图像或文本|	CLIP|
|视频理解|	视频|	文本摘要或分类标签|	VideoCLIP、TimeSformer|
|音视频多模态|	视频 + 音频|	多任务输出（转写+理解）	|Whisper + Vision Transformer|

In [13]:
from transformers import AutoModelForVisualQuestionAnswering, AutoProcessor
from PIL import Image
import torch

model_name = "dandelin/vilt-b32-finetuned-vqa"

# 加载模型和处理器
model = AutoModelForVisualQuestionAnswering.from_pretrained(model_name)
processor = AutoProcessor.from_pretrained(model_name)

# 读取图像和问题
image = Image.open("img/object.jpg")
question = "How many cats are there?"

# 将图像和问题编码
inputs = processor(image, question, return_tensors="pt")

# 前向传播（禁用梯度计算，加速）
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测的答案索引（logits 中最高分）
pred_idx = outputs.logits.argmax(-1).item()

# 从模型的配置中获取对应的答案文本
answer = model.config.id2label[pred_idx]

print(f"Answer: {answer}")

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 46141.48it/s]


Answer: 2


# 2. 文档问答（Document Question Answering）
> 提前安装  PIL, pytesseract,  PyTorch, transformers

> 安装 pytesseract Python 包：`pip install pytesseract`

> 安装 Tesseract OCR 引擎：`apt-get install tesseract-ocr`

安装完成后要重启 jupyter lab 的python内核。才可以测试生效

`OCR 全称是 Optical Character Recognition（光学字符识别），是一种将图像中的文字转换为可编辑文本的技术。`

In [1]:
import pytesseract
from PIL import Image

# 测试 OCR
image = Image.open("img/invoice.png")
text = pytesseract.image_to_string(image)
print("OCR 提取的文本:", text)


OCR 提取的文本: INVOICE

East Repair Inc.
1912 Harvest Lane
New York, NY 12210

BILLTO SHIP TO INVOICE # us-001
John Smith John Smith INVOICE DATE 1110272019
2 Court Square 3787 Pineview Drive Pose

New York, NY 12210 Cambridge, MA 12210 oy 2312/2019
DUE DATE 26/02/2019

ay DESCRIPTION UNIT PRICE AMOUNT

1 Front and rear brake cables 100.00 100.00

2 Newset of pedal arms 15.00 30.00

3 Labor Shrs 5.00 1.00

Subtotal 145.00

Sales Tax 6.25% 9.06

TOTAL $154.06

Smith

TERMS & CONDITIONS

Payment is due within 15 days
d hank you Plate mate cack pate: Est Rep



In [2]:
## 测试文档问答
from transformers import pipeline
from PIL import Image

# 如果在 Windows 上，可能需要指定 tesseract 的路径
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

nlp = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
)

image = Image.open("img/invoice.png")

res = nlp(
    image=image,
    question="What is the invoice number?"
)

print(res)

D:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 205/205 [00:00<00:00, 12023.44it/s]


[{'score': 0.4091435968875885, 'answer': 'us-001', 'start': 16, 'end': 16}]
